In [ ]:
# %% [markdown]
# # Stage 1 – 数据预处理与特征工程
# 
# 目标：清洗原始数据，构建特征，分层划分训练/测试集，标准化，保存处理后的数据。

# %% [markdown]
# ## 1. 导入库与配置

# %%
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os

# 设置路径（请根据实际存放位置修改）
DATA_RAW = '../data/Gaming_Academic_Performance.csv'
PROCESSED_DIR = '../data/processed/'
os.makedirs(PROCESSED_DIR, exist_ok=True)

# %%
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# %% [markdown]
# ## 2. 加载原始数据

# %%
df = pd.read_csv(DATA_RAW)
print(f"原始数据形状: {df.shape}")
print(f"列名: {df.columns.tolist()}\n")
print("前5行预览:")
df.head()

# %%
print("数据类型：")
print(df.dtypes)
print("\n缺失值统计：")
print(df.isnull().sum())
print(f"\n重复行数: {df.duplicated().sum()}")

# %% [markdown]
# ## 3. 描述性统计

# %%
df.describe()

# %% [markdown]
# ## 4. 分类变量唯一值检查

# %%
print("gender 唯一值:", df['gender'].unique())
print("gaming_genre 唯一值:", df['gaming_genre'].unique())
print("stress_level 唯一值:", df['stress_level'].unique())

# %% [markdown]
# ## 5. 异常处理与逻辑校验

# %%
def cap_outliers(df, cols):
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df[col] = df[col].clip(lower, upper)
    return df

num_cols = ['gaming_hours', 'study_hours', 'sleep_hours', 'attendance',
            'social_activity', 'device_usage', 'reaction_time_ms',
            'addiction_score', 'grades']

df_before = df.copy()
df = cap_outliers(df, num_cols)

for col in num_cols:
    before_min, before_max = df_before[col].min(), df_before[col].max()
    after_min, after_max = df[col].min(), df[col].max()
    if (before_min != after_min) or (before_max != after_max):
        print(f"{col}: 截尾前 [{before_min:.2f}, {before_max:.2f}] → 截尾后 [{after_min:.2f}, {after_max:.2f}]")

# %%
inconsistent = (df['gaming_hours'] > df['device_usage']).sum()
print(f"gaming_hours > device_usage 的行数: {inconsistent}")
print(f"grades 范围: [{df['grades'].min():.2f}, {df['grades'].max():.2f}]")
print(f"gaming_hours 范围: [{df['gaming_hours'].min():.2f}, {df['gaming_hours'].max():.2f}]")

# %% [markdown]
# ## 6. 分类变量编码

# %%
df = pd.get_dummies(df, columns=['gender'], prefix='gender', drop_first=True)
print("One-hot 编码后新增列:", [c for c in df.columns if 'gender_' in c])

# %%
genre_dummies = pd.get_dummies(df['gaming_genre'], prefix='genre')
df = pd.concat([df, genre_dummies], axis=1).drop('gaming_genre', axis=1)
print("游戏类型独热编码列:", genre_dummies.columns.tolist())

# %%
stress_map = {'Low': 0, 'Medium': 1, 'High': 2}
df['stress_level_ord'] = df['stress_level'].map(stress_map)
df = df.drop('stress_level', axis=1)
print("压力水平编码映射:", stress_map)
print("编码后分布:\n", df['stress_level_ord'].value_counts().sort_index())

# %% [markdown]
# ## 7. 特征工程

# %%
df['gaming_hours_sq'] = df['gaming_hours'] ** 2
df['gaming_hours_x_FPS'] = df['gaming_hours'] * df['genre_FPS']
df['gaming_hours_x_RPG'] = df['gaming_hours'] * df['genre_RPG']
df['gaming_study_ratio'] = df['gaming_hours'] / (df['study_hours'] + 0.01)
df['total_load'] = df['gaming_hours'] + df['study_hours']

print("新增特征:", ['gaming_hours_sq', 'gaming_hours_x_FPS', 'gaming_hours_x_RPG', 
                    'gaming_study_ratio', 'total_load'])

# %%
print(f"\n当前总特征数（含目标变量 grades）: {df.shape[1]}")
df.head(3)

# %% [markdown]
# ## 8. 划分训练集和测试集（分层抽样）

# %%
X = df.drop(['student_id', 'grades'], axis=1)
y = df['grades']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=df['stress_level_ord']
)

print(f"训练集特征形状: {X_train.shape}, 测试集特征形状: {X_test.shape}")
print(f"训练集目标形状: {y_train.shape}, 测试集目标形状: {y_test.shape}")

# %%
print("\n原始数据 stress_level_ord 分布:")
print(df['stress_level_ord'].value_counts(normalize=True).sort_index())
print("\n训练集 stress_level_ord 分布:")
print(X_train['stress_level_ord'].value_counts(normalize=True).sort_index())
print("\n测试集 stress_level_ord 分布:")
print(X_test['stress_level_ord'].value_counts(normalize=True).sort_index())

# %% [markdown]
# ## 9. 保存未缩放的数据

# %%
X_train.to_csv(os.path.join(PROCESSED_DIR, 'X_train.csv'), index=False)
X_test.to_csv(os.path.join(PROCESSED_DIR, 'X_test.csv'), index=False)
y_train.to_csv(os.path.join(PROCESSED_DIR, 'y_train.csv'), index=False)
y_test.to_csv(os.path.join(PROCESSED_DIR, 'y_test.csv'), index=False)
print(f"已保存未缩放数据至 {PROCESSED_DIR}")

# %% [markdown]
# ## 10. 特征标准化

# %%
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)

X_train_scaled_df.to_csv(os.path.join(PROCESSED_DIR, 'X_train_scaled.csv'), index=False)
X_test_scaled_df.to_csv(os.path.join(PROCESSED_DIR, 'X_test_scaled.csv'), index=False)
print("已保存标准化后的特征数据")

# %%
with open(os.path.join(PROCESSED_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
print("已保存 StandardScaler 对象至 scaler.pkl")

# %%
print(f"训练集标准化后均值（前5个特征）: {X_train_scaled_df.iloc[:, :5].mean().values}")
print(f"训练集标准化后标准差（前5个特征）: {X_train_scaled_df.iloc[:, :5].std().values}")

# %% [markdown]
# ## 11. 完成

# %%
print("Stage 1 数据处理完成！生成文件列表：")
for f in os.listdir(PROCESSED_DIR):
    print(f"  - {f}")